In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvZml0c2wvZGV2L3NpbXBsZXItbGV0dGVyYm94ZC9RdWFydG8='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/importlib/_bootstrap.py": 1771974507.315756, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/importlib/_bootstrap_external.py": 1771974507.3187559, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/zipimport.py": 1771974506.3617537, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/codecs.py": 1771974506.059753, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/encodings/aliases.py": 1771974506.7167544, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/encodings/__init__.py": 1771974506.7117546, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/encodings/utf_8.py": 1771974507.0237553, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/abc.py": 1771974506.0257528, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/stat.py": 1771974506.2887535, "/home/fitsl/miniconda3/envs/simple-tropeogram/lib/python3.14/_collections_abc.py": 1771

In [2]:
import sys, os
from pathlib import Path

_project_root = Path.cwd()
while not (_project_root / "src").exists():
    _project_root = _project_root.parent
os.chdir(_project_root)

In [3]:
from pathlib import Path
import random
from itertools import product

import srt
import pandas as pd
import chardet
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt  

from src.hedonometer import parse_text, import_hedonometer_as_dict

In [4]:
# constants
plt.style.use(["ggplot"])
IMG_WIDTH = 12
sns.set_palette("muted")

In [5]:
# srt_file = Path(
#     "Data/subtitles/Fellowship Of The Ring 2001 [Extended Edition] SDH-en.srt"
# )
# trope_file = Path("Data/trope_time_series/Fellowship_of_the_Ring_filled_copy.csv")
# movie_title = "FoTR"
# top_genres = ["Action", "Adventure", "Fantasy"]

srt_file = Path("Data/subtitles/Clueless.1995.srt")
trope_file = Path("Data/trope_time_series/clueless_tropes.csv")
movie_title = "Clueless"
top_genres = ["Comedy", "Action", "Drama"]

In [6]:
def parse_timestamp(t):
    if pd.isna(t):
        return np.nan
    # already a timedelta
    if hasattr(t, "total_seconds"):
        return t.total_seconds()
    t = str(t).strip()
    if ":" in t:
        parts = [int(p) for p in t.split(":")]
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    if "." in t:
        parts = [int(p) for p in t.split(".")]
        if len(parts) == 3:
            return parts[0] * 3600 + parts[1] * 60 + parts[2]
        elif len(parts) == 2:
            return parts[0] * 60 + parts[1]
    try:
        return float(t)
    except Exception as e:
        print(f"Due to {e}, casting {t} to nan")
        return np.nan

In [7]:
with open(srt_file, "r") as f:
    subtitle_generator = srt.parse(f.read())
subtitles = list(subtitle_generator)
subtitle_df = pd.DataFrame(
    [{"start": s.start, "end": s.end, "content": s.content} for s in subtitles]
)
subtitle_df["start"] = subtitle_df["start"].apply(parse_timestamp)
subtitle_df["end"] = subtitle_df["end"].apply(parse_timestamp)
subtitle_df = subtitle_df.sort_values(by="start", ascending=True)
subtitle_df.head()

,start,end,content
0,52.261,56.097,"Okay, you're probably going, ""ls this\na Noxze..."
1,56.265,60.727,"But seriously, I actually have\na way normal l..."
2,61.145,65.314,"I get up, I brush my teeth,\nand I pick out my..."
3,84.334,88.004,Daddy's a litigator.\nThose are the scariest k...
4,88.088,90.923,"Even Lucy, our maid, is terrified of him."


In [8]:
subtitle_df["content"] = subtitle_df["content"].str.replace("\n", " ")
subtitle_df["content"] = subtitle_df["content"].str.replace("<i>", " ")
subtitle_df["content"] = subtitle_df["content"].str.replace("</i>", " ")
subtitle_df["content"] = subtitle_df["content"].apply(parse_text)

In [9]:
subtitle_df["wordcount"] = subtitle_df["content"].str.len()
subtitle_df.head()

,start,end,content,wordcount
0,52.261,56.097,"[Okay, ,, you're, probably, going, ,, "", ls, t...",16
1,56.265,60.727,"[But, seriously, ,, I, actually, have, a, way,...",15
2,61.145,65.314,"[I, get, up, ,, I, brush, my, teeth, ,, and, I...",17
3,84.334,88.004,"[Daddy, 's, a, litigator, ., Those, are, the, ...",13
4,88.088,90.923,"[Even, Lucy, ,, our, maid, ,, is, terrified, o...",11


In [10]:
with open(trope_file, "rb") as f:
    encoding = chardet.detect(f.read())["encoding"]
trope_df = pd.read_csv(trope_file, encoding=encoding)
trope_df["start"] = trope_df["Start Time"].apply(parse_timestamp)
trope_df["end"] = trope_df["End Time"].apply(parse_timestamp)
trope_df["Camel Trope"] =  trope_df['Trope'].str.replace(" ", "") 
trope_df.head()

,Trope,Description,Inverted?/Defied?,Averted/Subverted?,Rough Occurence in movie,Background?,Setups?,Start Time,End Time,Unnamed: 9,Unnamed: 10,start,end,Camel Trope
0,The '90s,An odd but valid example — the whole movie se...,No,No,Whole Movie,Yes,No,0,5833,NaN,NaN,0.0,5833.0,The'90s
1,Actually Pretty Funny,Even though Heather's his girlfriend (at the ...,NaN,NaN,NaN,NaN,NaN,2661,2669,NaN,NaN,2661.0,2669.0,ActuallyPrettyFunny
2,Adaptation Title Change,Clueless is loosely based on Emma .,No,No,Whole Movie,Yes,No,0,5833,NaN,NaN,0.0,5833.0,AdaptationTitleChange
3,Adaptational Angst Downgrade,The scene in which Cher carelessly offends Lu...,No,No,NaN,NaN,NaN,4302,4323,NaN,NaN,4302.0,4323.0,AdaptationalAngstDowngrade
4,Adapted Out,"Due to Christian being gay, there is no Jane ...",NaN,NaN,NaN,NaN,NaN,3955,3989,NaN,NaN,3955.0,3989.0,AdaptedOut


In [11]:
trope_df["Description"] = trope_df["Description"].apply(parse_text)
trope_df["wordcount"] = trope_df["Description"].str.len()
trope_df["wordcount"].sum()

np.float64(5563.0)

In [12]:
def assign_happiness(data, lens_size, labmt: dict )-> np.ndarray:
    happiness_scores = np.array([labmt.get(word, np.nan) for word in data])
    happiness_scores = np.where(
        (happiness_scores < 5 - lens_size) | (happiness_scores > 5 + lens_size),
        happiness_scores,
        np.nan,
    )
    return happiness_scores

In [13]:
def sliding_window_scores(
    window_start,
    window_end,
    lens_size,
    labmt,
    subs: pd.DataFrame,
    tropes: pd.DataFrame,
):
    sub_tokens = []
    for _, row in subs.iterrows():
        if window_start <= row["start"] < window_end and isinstance(
            row["content"], list
        ):
            sub_tokens += row["content"]

    seen_tropes = set()
    trope_tokens = []
    for _, row in tropes.iterrows():
        key = (row["start"], row["end"])
        if (
            key not in seen_tropes
            and row["start"] < window_end
            and row["end"] > window_start
        ):
            trope_tokens += row["Description"]
            seen_tropes.add(key)

    sub_hs = assign_happiness(sub_tokens, lens_size, labmt)
    trope_hs = assign_happiness(trope_tokens, lens_size, labmt)
    return sub_hs, trope_hs

In [14]:
def aggregate_scores(sub_hs, trope_hs, alpha):
    sub_score = np.nan if np.all(np.isnan(sub_hs)) else np.nanmean(sub_hs)
    trope_score = np.nan if np.all(np.isnan(trope_hs)) else np.nanmean(trope_hs)

    num = np.nansum(sub_hs) + alpha * np.nansum(trope_hs)
    den = np.sum(~np.isnan(sub_hs)) + alpha * np.sum(~np.isnan(trope_hs))
    comb_score = num / den if den > 0 else np.nan
    return sub_score, trope_score, comb_score

In [15]:
def hedonometer_score(subs, tropes, lens_size, alpha, step_size=30):
    labmt = import_hedonometer_as_dict()
    end_time = max(subs["end"].max(), tropes["end"].max())
    results = []
    for start in np.arange(0, end_time, step_size):
        end = start + 4 * 60
        sub_hs, trope_hs = sliding_window_scores(
            start, end, lens_size, labmt, subs, tropes
        )
        sub_score, trope_score, comb_score = aggregate_scores(sub_hs, trope_hs, alpha)
        results.append(
            {
                "start": start,
                "end": end,
                "Subtitle": sub_score,
                "Trope": trope_score,
                "Combined": comb_score,
            }
        )
    return pd.DataFrame(results)

In [16]:
def plot_hedonometer(subs, tropes, lens_size, alpha, movie, ax=None):
    if not ax:
        _, ax = plt.subplots(figsize=(12, IMG_WIDTH))

    df = hedonometer_score(subs, tropes, lens_size, alpha, step_size=30)
    melted = df.melt(
        id_vars="start",
        value_vars=["Subtitle", "Trope", "Combined"],
        var_name="series",
        value_name="happiness",
    )
    sns.lineplot(
        data=melted,
        x="start",
        y="happiness",
        hue="series",
        ax=ax,
    )
    ax.set_xlabel("Time")
    ax.set_ylabel("Happiness")
    ax.set_title(
        rf"Average Happiness Over Time for {movie} with lens=${lens_size}, \alpha={alpha}$"
    )

In [17]:
alphas = [0.5, 0.75, 1]
lens_sizes = [1, 1.5]

for alpha, lens_size in product(alphas, lens_sizes):
    plot_hedonometer(subtitle_df, trope_df, lens_size, alpha, movie_title)

<Figure size 3600x3600 with 1 Axes>

<Figure size 3600x3600 with 1 Axes>

<Figure size 3600x3600 with 1 Axes>

<Figure size 3600x3600 with 1 Axes>

<Figure size 3600x3600 with 1 Axes>

<Figure size 3600x3600 with 1 Axes>

In [18]:
def combine_frames(subs: pd.DataFrame, tropes: pd.DataFrame):
    combined = subs.copy(deep=True)
    n_rows = subs.shape[0]
    combined["active tropes"] = [[] for _ in range(n_rows)]
    combined["trope content"] = [[] for _ in range(n_rows)]

    unplaced = []

    for _, trow in tropes.iterrows():
        placed = False
        for idx, srow in subs.iterrows():
            if trow.start < srow.end and trow.end > srow.start:
                placed = True
                combined.at[idx, "active tropes"] += [trow["Camel Trope"]]
                combined.at[idx, "trope content"] += trow["Description"]
        if not placed:
            unplaced.append({
                "start": trow.start,
                "end": trow.end,
                "content": [],
                "active tropes": [trow["Trope"]],
                "trope content": trow["Description"],
            })

    full = pd.concat([combined, pd.DataFrame(unplaced)], ignore_index=True)
    return full.sort_values("start").reset_index(drop=True)   

In [19]:
def build_tf_idf_matrix(min_movies=5):
    df = pd.read_csv("Data/2020_genre_counts_by_trope.csv")
    genres = df.columns[2:29].tolist()
    num_genres = len(genres)

    # drop tropes too sparse to give reliable genre signal
    df = df[df["Number_movies"] >= min_movies].copy()

    df_tropes = df[genres]  # original counts, used for IDF

    # smoothed IDF: log((1 + N) / (1 + df_t)) + 1
    # prevents single-genre tropes from getting IDF = log(27) ~ 3.3 and dominating everything
    df["idf"] = np.log((1 + num_genres) / (1 + (df_tropes > 0).sum(axis=1))) + 1

    # TF: fraction of this trope's genre signal pointing at each genre
    row_sums = df[genres].sum(axis=1)
    df[genres] = df[genres].divide(row_sums.where(row_sums > 0, 1), axis=0)

    # combine
    df[genres] = df[genres].multiply(df["idf"], axis=0)
    df.drop(columns=["idf"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    df["Trope"] = df["Trope"].str.strip()
    df = df[["Trope"] + genres]
    df.dropna(how="all", inplace=True)
    df.set_index("Trope", inplace=True)
    df.to_csv("Data/Cleaned-tf_idf-matrix.csv")
    return df


tf_idf = build_tf_idf_matrix()
tf_idf.head()

,Action,Adult,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,...,Mystery,News,Romance,Sci-Fi,Short,Sport,Thriller,War,Western,\N
Trope,,,,,,,,,,,,,,,,,,,,,
ABNegative,0.273674,0.0,0.068418,0.0,0.000000,0.205255,0.205255,0.000000,0.547348,0.068418,...,0.068418,0.0,0.136837,0.000000,0.000000,0.0,0.136837,0.000000,0.0,0.068418
ABirthdayNotABreak,0.264554,0.0,0.105822,0.0,0.052911,0.105822,0.211643,0.052911,0.264554,0.000000,...,0.158733,0.0,0.000000,0.052911,0.000000,0.0,0.158733,0.052911,0.0,0.000000
ABloodyMess,0.173184,0.0,0.173184,0.0,0.000000,0.461824,0.057728,0.000000,0.346368,0.057728,...,0.115456,0.0,0.230912,0.057728,0.000000,0.0,0.115456,0.000000,0.0,0.000000
ABoyAndHisX,0.102628,0.0,0.307883,0.0,0.000000,0.153941,0.051314,0.000000,0.513138,0.410511,...,0.051314,0.0,0.000000,0.051314,0.051314,0.0,0.051314,0.000000,0.0,0.000000
ACappella,0.000000,0.0,0.000000,0.0,0.000000,0.795431,0.000000,0.198858,0.000000,0.000000,...,0.000000,0.0,0.596574,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000


In [20]:
def assign_generic_scores(tropes, tf_idf):
    if not tropes:
        return {g: 0.0 for g in tf_idf.columns}
    valid = [t for t in tropes if t in tf_idf.index]
    raw = tf_idf.loc[valid].sum()
    total = raw.sum()
    return (raw / total if total > 0 else raw).to_dict()

In [21]:
def build_genre_columns(combined, tf_idf):
    scores = combined['active tropes'].apply(lambda tropes: assign_generic_scores(tropes, tf_idf))
    return pd.concat([combined[['start', 'end', 'active tropes']], pd.DataFrame(scores.tolist(), index=combined.index)], axis =1)

In [22]:
combined = combine_frames(subtitle_df, trope_df)
combined = build_genre_columns(combined , tf_idf)
combined.head(20)

,start,end,active tropes,Action,Adult,Adventure,Animation,Biography,Comedy,Crime,...,Mystery,News,Romance,Sci-Fi,Short,Sport,Thriller,War,Western,\N
0,0.000,0.000,[Status Cell Phone],0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000,0.000,[Stern Teacher],0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000,0.000,[nan],0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000,0.000,[The Stoner],0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,52.261,56.097,"[The'90s, AdaptationTitleChange, AlphaBitch, A...",0.091803,0.0,0.088428,0.005665,0.005994,0.200048,0.039515,...,0.019845,0.0,0.094008,0.034937,0.005736,0.014802,0.043167,0.005494,0.007045,0.000111
5,56.265,60.727,"[The'90s, AdaptationTitleChange, AlphaBitch, A...",0.091803,0.0,0.088428,0.005665,0.005994,0.200048,0.039515,...,0.019845,0.0,0.094008,0.034937,0.005736,0.014802,0.043167,0.005494,0.007045,0.000111
6,61.145,65.314,"[The'90s, AdaptationTitleChange, AlphaBitch, A...",0.094703,0.0,0.088194,0.005427,0.008411,0.191448,0.043752,...,0.020250,0.0,0.090238,0.035691,0.005698,0.013681,0.045089,0.006151,0.007448,0.000100
7,84.334,88.004,"[The'90s, AdaptationTitleChange, AlphaBitch, A...",0.094703,0.0,0.088194,0.005427,0.008411,0.191448,0.043752,...,0.020250,0.0,0.090238,0.035691,0.005698,0.013681,0.045089,0.006151,0.007448,0.000100
8,88.088,90.923,"[The'90s, AdaptationTitleChange, AlphaBitch, A...",0.094703,0.0,0.088194,0.005427,0.008411,0.191448,0.043752,...,0.020250,0.0,0.090238,0.035691,0.005698,0.013681,0.045089,0.006151,0.007448,0.000100
9,91.049,94.927,"[The'90s, AdaptationTitleChange, AlphaBitch, A...",0.094703,0.0,0.088194,0.005427,0.008411,0.191448,0.043752,...,0.020250,0.0,0.090238,0.035691,0.005698,0.013681,0.045089,0.006151,0.007448,0.000100


In [23]:
def plot_all(subs, tropes, lens_size, alpha, movie, top_genres, tf_idf):
    fig, axes = plt.subplots(2, 2, figsize=(IMG_WIDTH * 2, IMG_WIDTH))
    fig.suptitle(f"{movie} — lens={lens_size}, α={alpha}")

    plot_hedonometer(subs, tropes, lens_size, alpha, movie, ax=axes[0, 0])
    combined = combine_frames(subs, tropes)
    genre_df = build_genre_columns(combined, tf_idf)

    for ax, genre in zip(axes.flat[1:], top_genres):
        sns.lineplot(data=genre_df, x="start", y=genre, ax=ax, color="black")
        ax.set_title(genre)
        ax.set_xlabel("Time")
        ax.set_ylabel("Genre Score")
        ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.show()


plot_all(
    subtitle_df,
    trope_df,
    lens_size=1.5,
    alpha=1,
    movie=movie_title,
    top_genres=top_genres,
    tf_idf=tf_idf,
)

<Figure size 7200x3600 with 4 Axes>